# 🇮🇳 Mistral-7B QLoRA Evaluation: Perplexity & Response Generation

This notebook provides an end-to-end evaluation pipeline for our fine-tuned Hinglish Mistral-7B model:
1. **Quantitative Language Modeling Assessment:** Calculates **Perplexity (PPL)** on the test split for both the **Base Model (Before)** and **LoRA Fine-Tuned Model (After)** using in-memory adapter toggling (`model.disable_adapter()`).
2. **Benchmark Response Generation:** Randomly samples **200 prompts** from the test dataset and generates responses from both Base and LoRA models.
3. **Unified Evaluation Suite (`HinglishEvaluator`):** Computes **Script Purity** (Latin % and Devanagari character leakage) and multi-dimensional **LLM-as-a-Judge** scoring (Coherence, Helpfulness, Hinglish Naturalness, and Blind Pairwise Win Rate) via Google Gemini API.

## 1. ⚙️ Setup & Dependencies

Ensure `bitsandbytes`, `transformers`, `peft`, and `google-generativeai` are installed.

In [ ]:
# Upgrade libraries if needed on Kaggle
!pip install -q -U "bitsandbytes>=0.43.0" "transformers>=4.40.0" "peft>=0.10.0" datasets pyyaml tqdm google-generativeai

import os
import sys
import json
import random
import torch
import yaml
from tqdm.auto import tqdm
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Add repository root to path so evaluation module is importable
if os.path.exists("peft-recipes-lab"):
    sys.path.append("peft-recipes-lab")
sys.path.append(".")
sys.path.append("..")

print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## 2. 📂 Load Configuration and Test Data

We load the test dataset using the exact same split parameters (`test_size=0.1`, `seed=42`) used during training.

In [ ]:
CHECKPOINT_PATH = "/kaggle/working/mistral-lora-outputs/checkpoint-4000"
if not os.path.exists(CHECKPOINT_PATH):
    CHECKPOINT_PATH = "./mistral-lora-outputs/checkpoint-4000"

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
DATASET_PATH = "theguywithblacktie/hinglish-conversations"
DATASET_CONFIG = "small"
SEED = 42
TEST_SIZE = 0.1
NUM_GEN_SAMPLES = 200

print(f"Loading test split from {DATASET_PATH} (config={DATASET_CONFIG})...")
raw_dataset = load_dataset(DATASET_PATH, DATASET_CONFIG, split="train")
split_dict = raw_dataset.train_test_split(test_size=TEST_SIZE, seed=SEED)
test_data = split_dict["test"]

print(f"Total Test Samples: {len(test_data)}")
print("Sample item from test set:", test_data[0])

## 3. 🧠 Load 4-bit Quantized Model with LoRA Adapter

We load the base model in 4-bit (NF4) and attach the trained LoRA adapter. We can toggle between **Base Model** and **Fine-Tuned Model** at zero cost using `model.disable_adapter()`.

In [ ]:
print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print(f"Loading Base Model ({MODEL_ID}) in 4-bit...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

print(f"Attaching LoRA Adapter from {CHECKPOINT_PATH}...")
model = PeftModel.from_pretrained(base_model, CHECKPOINT_PATH)
model.eval()
print("✅ Model and adapter loaded successfully!")

## 4. 📉 Metric 1: Perplexity Evaluation (Cross-Entropy Loss)

Perplexity measures the model uncertainty on the ground-truth test data.
* Formats test data with the Mistral chat template.
* Evaluates cross-entropy loss across batches in single parallel forward passes (no text generation required).
* Compares **Base Mistral-7B (adapter disabled)** vs. **LoRA Fine-Tuned Model**.

In [ ]:
from evaluation.perplexity import PerplexityEvaluator

def format_chat(sample):
    messages = [
        {"role": "user", "content": sample["input"]},
        {"role": "assistant", "content": sample["output"]}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

print("Formatting test set for Perplexity evaluation...")
formatted_test = test_data.map(format_chat, desc="Applying chat template")

PPL_SAMPLES = min(1000, len(formatted_test))
print(f"Evaluating Perplexity on {PPL_SAMPLES} samples...")

ppl_evaluator = PerplexityEvaluator()
ppl_results = ppl_evaluator.compare_models(
    lora_model=model,
    tokenizer=tokenizer,
    dataset=formatted_test,
    base_model=None,  # Automatically evaluates base model via model.disable_adapter()
    max_samples=PPL_SAMPLES,
    max_seq_length=512,
    batch_size=8
)

print("\n" + "="*60)
print("📊 PERPLEXITY EVALUATION RESULTS")
print("="*60)
print(f"Base Mistral-7B PPL:       {ppl_results['base_model']['perplexity']:.2f} (Loss: {ppl_results['base_model']['loss']:.4f})")
print(f"LoRA Fine-Tuned PPL:      {ppl_results['lora_model']['perplexity']:.2f} (Loss: {ppl_results['lora_model']['loss']:.4f})")
print(f"Perplexity Reduction:     {ppl_results['perplexity_reduction_pct']:.2f}%")
print("="*60)

os.makedirs("/kaggle/working/eval_results", exist_ok=True)
with open("/kaggle/working/eval_results/perplexity_summary.json", "w") as f:
    json.dump(ppl_results, f, indent=2)
print("💾 Saved perplexity summary to /kaggle/working/eval_results/perplexity_summary.json")

## 5. 🤖 Response Generation for 200 Random Test Prompts

We randomly sample **200 prompts** from the test dataset and generate responses for both:
1. `response_base`: Base Mistral-7B generation (`with model.disable_adapter():`)
2. `response_lora`: Fine-Tuned QLoRA generation

Generating 200 pairs takes **~8 to 12 minutes** on a Kaggle GPU.

In [ ]:
random.seed(SEED)
sample_indices = random.sample(range(len(test_data)), NUM_GEN_SAMPLES)
eval_subset = [test_data[i] for i in sample_indices]

def generate_text(prompt_text, use_adapter=True):
    messages = [{"role": "user", "content": prompt_text}]
    prompt_str = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_str, return_tensors="pt").to(model.device)
    
    gen_kwargs = dict(
        **inputs,
        max_new_tokens=256,
        temperature=0.3,
        repetition_penalty=1.05,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id
    )
    
    with torch.no_grad():
        if use_adapter:
            outputs = model.generate(**gen_kwargs)
        else:
            with model.disable_adapter():
                outputs = model.generate(**gen_kwargs)
                
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

generations = []
print(f"Generating responses for {NUM_GEN_SAMPLES} test samples...")

for item in tqdm(eval_subset, desc="Generating Base vs LoRA"):
    prompt = item["input"]
    ground_truth = item["output"]
    
    resp_base = generate_text(prompt, use_adapter=False)
    resp_lora = generate_text(prompt, use_adapter=True)
    
    generations.append({
        "prompt": prompt,
        "ground_truth": ground_truth,
        "response_base": resp_base,
        "response_lora": resp_lora
    })

output_json_path = "/kaggle/working/eval_results/eval_generations_200.json"
with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(generations, f, ensure_ascii=False, indent=2)

print(f"\n💾 Saved {len(generations)} generated response pairs to {output_json_path}!")

## 6. 👀 Qualitative Preview: 3 Random Generated Samples

Let's visually inspect 3 sample pairs directly in the notebook before running the full evaluation metrics.

In [ ]:
from IPython.display import display, HTML

html_out = "<h3>🔍 Preview: Base Model vs. LoRA Fine-Tuned (Mistral-7B)</h3>"
preview_samples = random.sample(generations, min(3, len(generations)))

for i, sample in enumerate(preview_samples, 1):
    p = sample["prompt"]
    b = sample["response_base"]
    l = sample["response_lora"]
    html_out += f"""
    <div style="border: 1px solid #ccc; border-radius: 8px; padding: 12px; margin-bottom: 16px; font-family: sans-serif;">
        <div style="font-weight: bold; color: #1e3a8a; margin-bottom: 8px;">💬 Prompt {i}: {p}</div>
        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 12px;">
            <div style="background: #fff1f2; border: 1px solid #fecdd3; padding: 10px; border-radius: 6px;">
                <div style="color: #be123c; font-weight: bold;">❌ Base Mistral-7B (Before)</div>
                <div style="font-size: 13px; margin-top: 6px; white-space: pre-wrap;">{b}</div>
            </div>
            <div style="background: #f0fdf4; border: 1px solid #bbf7d0; padding: 10px; border-radius: 6px;">
                <div style="color: #15803d; font-weight: bold;">✅ LoRA Fine-Tuned (After)</div>
                <div style="font-size: 13px; margin-top: 6px; white-space: pre-wrap;">{l}</div>
            </div>
        </div>
    </div>
    """

display(HTML(html_out))

## 7. ⚖️ Evaluation via `HinglishEvaluator` (Script Purity + Gemini LLM Judge)

We now run [`HinglishEvaluator`](file:///mnt/d/peft-recipes-lab/evaluation/evaluator.py) on our 200 samples.
* **Script Purity:** Evaluates Latin % and Devanagari character bleed.
* **LLM-as-a-Judge (Gemini 1.5 Flash):** Evaluates Coherence, Helpfulness, Hinglish Naturalness, and Blind Pairwise Win Rate.

> **Note on Gemini API Key:** You can supply your key via Kaggle Secrets (`Add-ons -> Secrets -> GEMINI_API_KEY`), set `os.environ["GEMINI_API_KEY"]`, or paste it into the cell below. If left empty, the script will still compute the complete Script Purity evaluation.

In [ ]:
import os
from evaluation.evaluator import HinglishEvaluator
from IPython.display import display, Markdown

# 1. Retrieve Gemini API key from Kaggle Secrets, environment, or direct variable
gemini_api_key = os.environ.get("GEMINI_API_KEY", "")
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    gemini_api_key = user_secrets.get_secret("GEMINI_API_KEY") or gemini_api_key
except Exception:
    pass

# You can also paste your Gemini API key directly here if not using Kaggle Secrets:
# gemini_api_key = "your-api-key-here"

run_judge = bool(gemini_api_key and gemini_api_key.strip())
print(f"Gemini API Key configured: {run_judge}")
if not run_judge:
    print("ℹ️ No GEMINI_API_KEY provided. Running Script Purity only. Set the key to run LLM Judge.")

# 2. Initialize HinglishEvaluator
evaluator = HinglishEvaluator(
    judge_model_name="gemini-1.5-flash",
    judge_api_key=gemini_api_key if run_judge else None
)

# 3. Run Evaluation across all 200 samples
print(f"Evaluating {len(generations)} samples...")
report = evaluator.evaluate_generation_quality(
    test_samples=generations,
    run_llm_judge=run_judge
)

# 4. Generate and display the Markdown Summary Dashboard
summary_md = evaluator.generate_markdown_summary(report)
display(Markdown(summary_md))

# 5. Save report files to disk
report_md_path = "/kaggle/working/eval_results/evaluation_report.md"
with open(report_md_path, "w", encoding="utf-8") as f:
    f.write(summary_md)

report_json_path = "/kaggle/working/eval_results/evaluation_report.json"
with open(report_json_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(f"\n💾 Full Markdown report saved to: {report_md_path}")
print(f"💾 Full JSON report saved to:     {report_json_path}")